In [2]:
import numpy as np
from numpy.lib.stride_tricks import as_strided

In [3]:
def _pair(x):
    return tuple(x) if isinstance(x, (tuple, list)) else (x, x)

In [4]:
class Conv2d:
    def __init__(self, in_channels, out_channels, kernel_size,
                 stride=1, padding=0, dilation=1, bias=True, init='kaiming'):
        self.in_channels  = in_channels
        self.out_channels = out_channels
        self.kernel_size  = _pair(kernel_size)
        self.stride       = _pair(stride)
        self.padding      = _pair(padding)
        self.dilation     = _pair(dilation)
        self.use_bias     = bias

        kh, kw = self.kernel_size
        fan_in  = in_channels  * kh * kw
        fan_out = out_channels * kh * kw

        # ---------- 权重初始化 ----------
        # Kaiming(He)：假设后接 ReLU，std = sqrt(2 / fan_in)
        # Xavier(Glorot)：适合 tanh/sigmoid，std = sqrt(2 / (fan_in+fan_out))
        if   init == 'kaiming': std = np.sqrt(2.0 / fan_in)
        elif init == 'xavier' : std = np.sqrt(2.0 / (fan_in + fan_out))
        else                  : std = np.sqrt(1.0 / fan_in)
        self.weight      = (np.random.randn(out_channels, in_channels, kh, kw) * std).astype(np.float32)
        self.grad_weight = None

        # ---------- 偏置 ----------
        # 偏置默认置 0；若接 BN 可 bias=False。
        # 若想给 ReLU 一点“启动”余地，也可用 0.01 常数。
        if bias:
            self.bias      = np.zeros(out_channels, dtype=np.float32)
            self.grad_bias = None
        else:
            self.bias = self.grad_bias = None

        self._cache = {}

    # ---------- PyTorch 风格的小工具 ----------
    def __call__(self, x):        return self.forward(x)
    @property
    def params(self):             return [self.weight] + ([self.bias] if self.use_bias else [])
    @property
    def grads(self):              return [self.grad_weight] + ([self.grad_bias] if self.use_bias else [])
    def zero_grad(self):
        self.grad_weight = None
        self.grad_bias   = None

    # ---------- 前向 ----------
    def _pad(self, x):
        ph, pw = self.padding
        if ph == 0 and pw == 0:
            return x
        return np.pad(x, ((0,0),(0,0),(ph,ph),(pw,pw)), mode='constant')

    def _im2col(self, x):
        """x: (N, C, H, W) 已 padding。返回 cols: (N*out_h*out_w, C*kh*kw)"""
        N, C, H, W = x.shape
        kh, kw = self.kernel_size
        sh, sw = self.stride
        dh, dw = self.dilation
        out_h = (H - dh*(kh-1) - 1) // sh + 1
        out_w = (W - dw*(kw-1) - 1) // sw + 1

        sN, sC, sH, sW = x.strides
        # 用 stride_tricks 构造滑动视图，零拷贝
        view = as_strided(
            x,
            shape=(N, C, out_h, out_w, kh, kw),
            strides=(sN, sC, sH*sh, sW*sw, sH*dh, sW*dw),
        )
        # (N, out_h, out_w, C, kh, kw) -> (N*out_h*out_w, C*kh*kw)
        cols = view.transpose(0, 2, 3, 1, 4, 5).reshape(N*out_h*out_w, -1)
        return cols, out_h, out_w

    def forward(self, x):
        xp = self._pad(x)
        cols, out_h, out_w = self._im2col(xp)
        N = x.shape[0]

        W2 = self.weight.reshape(self.out_channels, -1)          # (O, C*kh*kw)
        out = cols @ W2.T                                        # (N*out_h*out_w, O)
        out = out.reshape(N, out_h, out_w, self.out_channels).transpose(0, 3, 1, 2)
        if self.use_bias:
            out = out + self.bias.reshape(1, -1, 1, 1)

        self._cache = dict(x_shape=x.shape, cols=cols, out_h=out_h, out_w=out_w)
        return out

    # ---------- 反向 ----------
    def _col2im(self, grad_cols, x_shape):
        """把 (N*out_h*out_w, C*kh*kw) 散射回 (N, C, H+2p, W+2p)"""
        N, C, H, W = x_shape
        kh, kw = self.kernel_size
        sh, sw = self.stride
        dh, dw = self.dilation
        ph, pw = self.padding
        out_h = (H + 2*ph - dh*(kh-1) - 1) // sh + 1
        out_w = (W + 2*pw - dw*(kw-1) - 1) // sw + 1

        g = grad_cols.reshape(N, out_h, out_w, C, kh, kw)
        g = g.transpose(0, 3, 4, 5, 1, 2)          # (N, C, kh, kw, out_h, out_w)

        gx = np.zeros((N, C, H + 2*ph, W + 2*pw), dtype=g.dtype)
        for i in range(kh):
            i_end = i*dh + (out_h-1)*sh + 1
            for j in range(kw):
                j_end = j*dw + (out_w-1)*sw + 1
                gx[:, :, i*dh:i_end:sh, j*dw:j_end:sw] += g[:, :, i, j]
        if ph or pw:
            gx = gx[:, :, ph:ph+H, pw:pw+W]
        return gx

    def backward(self, grad_out):
        c = self._cache
        N = c['x_shape'][0]
        cols = c['cols']

        g = grad_out.transpose(0, 2, 3, 1).reshape(-1, self.out_channels)

        # dW = grad_flat^T · cols
        self.grad_weight = (g.T @ cols).reshape(self.weight.shape)
        # db = sum over N,out_h,out_w
        if self.use_bias:
            self.grad_bias = g.sum(axis=0)

        # dX 通过 cols 回传后再 col2im
        W2 = self.weight.reshape(self.out_channels, -1)
        grad_cols = g @ W2
        return self._col2im(grad_cols, c['x_shape'])